In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile, f_classif
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
import joblib

In [2]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-gex")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.gex.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

output_dir = "/home/mariya/Data/pallas/models/gex"
random_seed = 42
n_jobs = 10
selected_features_path = os.path.join(output_dir, "selected_features.txt")
pipeline_path = os.path.join(output_dir, "pipeline.pkl")
hyperparams_path = os.path.join(output_dir, "hyperparams.pkl")

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

# Hardcoded variables
FINAL_N_ESTIMATORS=500
FEATURE_SELECTION_PERCENTILE=20

X_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-gex/features.gex.csv
y_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-gex/labels.csv
output_dir: /home/mariya/Data/pallas/models/gex
random_seed: 42
n_jobs: 10
selected_features_path: /home/mariya/Data/pallas/models/gex/selected_features.txt
pipeline_path: /home/mariya/Data/pallas/models/gex/pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/models/gex/hyperparams.pkl


In [3]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000288825,ENSG00000288859,ENSG00000288867,ENSG00000289549,ENSG00000289604,ENSG00000290292,ENSG00000291237,ENSG00000293137,ENSG00000293560,ENSG00000310560
id,,,,,,,,,,,,,,,,,,,,,
16-001,-0.863635,-1.117394,3.917742,3.683651,4.161335,4.600595,1.085004,1.574825,4.672992,5.301176,...,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-0.022777,-1.117394,5.335215
16-002,-0.565969,-1.117394,4.296498,2.197319,3.776878,3.791377,1.054676,2.747079,4.211614,5.478628,...,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-0.956718,-0.791161,5.775667
16-006,-0.816922,-1.117394,4.105142,3.755336,3.338865,4.226676,-0.285286,2.494947,3.913194,5.639054,...,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,0.435326,-1.117394,6.306637
16-008,3.970207,-1.117394,3.968898,4.143316,3.053780,5.292176,0.658723,2.907728,5.478920,5.885538,...,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,2.242500,-1.117394,7.983145
16-015,-0.109163,-1.117394,4.934040,2.949734,2.135491,5.193152,0.572912,2.607942,3.194204,4.137460,...,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-1.117394,-0.489496,-1.117394,5.465756
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-AB-2990,-1.117394,-1.117394,4.362424,4.083416,3.956593,4.169105,0.919650,3.562836,4.150124,5.139970,...,2.686098,3.236715,0.658384,-1.117394,-0.682002,4.532213,4.921797,-1.117394,0.231269,7.667080
TCGA-AB-2991,-1.117394,-0.621431,3.921344,3.232167,3.653022,0.541265,1.974768,3.646472,4.168509,6.748113,...,1.353648,0.489411,-1.117394,-1.117394,-1.117394,4.182676,4.374531,0.029213,0.551065,5.727486
TCGA-AB-2994,-1.117394,-1.117394,3.786989,4.087582,3.723462,3.436887,4.156968,3.927710,3.593320,4.889818,...,1.322593,0.878604,-1.117394,-1.117394,-1.117394,4.368065,4.179805,0.354355,-0.124385,6.072917


In [4]:
y_train = pd.read_csv(y_train_path, index_col=0)
y_train

,class
id,
16-001,"B-ALL, ETV6::RUNX1"
16-002,"B-ALL, ETV6::RUNX1"
16-006,"B-ALL, ETV6::RUNX1"
16-008,T-ALL
16-015,"B-ALL, high hyperdiploid"
...,...
TCGA-AB-2990,"AML, NPM1alt"
TCGA-AB-2991,"AML, APL-PML::RARA"
TCGA-AB-2994,"AML, APL-PML::RARA"


In [5]:
print("y_train class distribution:")
print(y_train.value_counts())

y_train class distribution:
class                   
B-ALL, high hyperdiploid    389
AML, KMT2A-r                365
B-ALL, ETV6::RUNX1          323
B-ALL, BCR::ABL1-like       261
AML, RUNX1::RUNX1T1         253
AML, NPM1alt                159
B-ALL, KMT2A-r              152
B-ALL, PAX5alt              141
T-ALL                       125
AML, CBFB::MYH11            117
B-ALL, DUX4-r               115
B-ALL, BCR::ABL1             97
B-ALL, TCF3::PBX1            88
B-ALL, hypodiploid           87
AML, CEBPAalt                79
AML, NUP98-r                 67
B-ALL, ZNF384-r              56
B-ALL, ETV6::RUNX1-like      43
B-ALL, MEF2D-r               40
B-ALL, iAMP21                36
AML, DEK::NUP214             31
B-ALL, PAX5 P80R             27
AML, CBFA2T3::GLIS2          25
Non-leukemic                 15
AML, APL-PML::RARA           14
B-ALL, NUTM1-r               13
B-ALL, MYC-r                 12
AML, ETV6-r                   8
AML, KAT6A-r                  8
B-ALL, HLF-r       

In [6]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(5)

In [7]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array

In [8]:
# Reduce memory usage by converting to float32
X_train = X_train.astype(np.float32)

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV

N_ITER = 70
N_REPEATS = 1
N_FOLDS_INITIAL = max(5, min(min_class_count, 10))

pipeline = Pipeline([
    ("feature_selection", SelectPercentile(
        score_func=f_classif
    )),
    ("rf", RandomForestClassifier(
        random_state=random_seed,
        n_jobs=1           # parallelism handled at search level
    ))
])

param_grid = {
    'feature_selection__percentile': [20],
    'rf__n_estimators': [50], # Set to fewer estimators for faster search; can be increased for final model
    'rf__max_depth': [20, 30, 40],
    # min_samples_leaf — hard upper bound of 2 given min class = 5
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': [0.15, 0.25, 0.3, 0.4],
    'rf__class_weight': ['balanced'],
}

cv = RepeatedStratifiedKFold(
    n_splits=N_FOLDS_INITIAL,
    n_repeats=N_REPEATS,
    random_state=random_seed
)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,             # parallelise at search level
    random_state=random_seed,
    verbose=2,
    refit=True
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (f1_macro):",   search.best_score_)

best_pipeline = search.best_estimator_

Fitting 5 folds for each of 24 candidates, totalling 120 fits


/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 24 is smaller than n_iter=70. Running 24 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END feature_selection__percentile=20, rf__class_weight=balanced, rf__max_depth=20, rf__max_features=0.15, rf__min_samples_leaf=1, rf__n_estimators=50; total time= 3.5min
[CV] END feature_selection__percentile=20, rf__class_weight=balanced, rf__max_depth=20, rf__max_features=0.15, rf__min_samples_leaf=1, rf__n_estimators=50; total time= 3.5min
[CV] END feature_selection__percentile=20, rf__class_weight=balanced, rf__max_depth=20, rf__max_features=0.15, rf__min_samples_leaf=2, rf__n_estimators=50; total time= 3.5min
[CV] END feature_selection__percentile=20, rf__class_weight=balanced, rf__max_depth=20, rf__max_features=0.15, rf__min_samples_leaf=2, rf__n_estimators=50; total time= 3.6min
[CV] END feature_selection__percentile=20, rf__class_weight=balanced, rf__max_depth=20, rf__max_features=0.15, rf__min_samples_leaf=2, rf__n_estimators=50; total time= 3.6min
[CV] END feature_selection__percentile=20, rf__class_weight=balanced, rf__max_depth=20, rf__max_features=0.15, rf__min_sample

In [10]:

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparams_path)

# Save features
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = X_train.columns[selected_mask].tolist()

# Write selected features to a text file
with open(selected_features_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

['/home/mariya/Data/pallas/models/gex/pipeline.pkl']

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
import joblib, os

N_REPEATS = 10

# Load and parse best params
with open(hyperparams_path, "rb") as f:
    best_params = joblib.load(f)

rf_params = {
    k.replace('rf__', ''): v
    for k, v in best_params.items()
    if k.startswith('rf__')
}
fs_percentile = best_params.get(
    'feature_selection__percentile',
    FEATURE_SELECTION_PERCENTILE
)
rf_params['n_estimators'] = FINAL_N_ESTIMATORS

print("Final RF parameters:", rf_params)
print("Feature selection percentile:", fs_percentile)

# Build final pipeline
final_pipeline = Pipeline([
    ("feature_selection", SelectPercentile(
        score_func=f_classif,
        percentile=fs_percentile
    )),
    ("rf", RandomForestClassifier(
        **rf_params,
        n_jobs=1,
        random_state=random_seed
    ))
])

# Cross-validate with train score to monitor gap
n_splits = max(5, min(min_class_count, 10))
cv = RepeatedStratifiedKFold(
    n_splits=n_splits,
    n_repeats=N_REPEATS,
    random_state=random_seed
)

cv_results = cross_validate(
    final_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

train_mean = cv_results['train_score'].mean()
train_std  = cv_results['train_score'].std()
val_mean   = cv_results['test_score'].mean()
val_std    = cv_results['test_score'].std()

print(f"Train F1-macro: {train_mean:.3f} ± {train_std:.3f}")
print(f"Val   F1-macro: {val_mean:.3f} ± {val_std:.3f}")
print(f"Gap:            {train_mean - val_mean:.3f}")

# Fit on all training data
final_pipeline.fit(X_train, y_train)


Final RF parameters: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.25, 'max_depth': 40, 'class_weight': 'balanced'}
Feature selection percentile: 20


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.


[CV] END .................................................... total time=66.7min
[CV] END .................................................... total time=67.8min
[CV] END .................................................... total time=67.8min
[CV] END .................................................... total time=67.8min
[CV] END .................................................... total time=67.9min
[CV] END .................................................... total time=68.1min
[CV] END .................................................... total time=68.2min
[CV] END .................................................... total time=68.3min
[CV] END .................................................... total time=68.3min
[CV] END .................................................... total time=68.7min
[CV] END .................................................... total time=68.9min
[CV] END .................................................... total time=69.2min
[CV] END ...................

[Parallel(n_jobs=-1)]: Done  13 tasks      | elapsed: 69.5min


[CV] END .................................................... total time=69.8min
[CV] END .................................................... total time=63.7min
[CV] END .................................................... total time=64.9min
[CV] END .................................................... total time=64.9min
[CV] END .................................................... total time=65.4min
[CV] END .................................................... total time=66.4min
[CV] END .................................................... total time=65.3min
[CV] END .................................................... total time=64.5min
[CV] END .................................................... total time=64.5min
[CV] END .................................................... total time=64.5min
[CV] END .................................................... total time=65.9min
[CV] END .................................................... total time=66.6min
[CV] END ...................

[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed: 246.5min finished


,steps,"[('feature_selection', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,score_func,<function f_c...x7e64cbc36d40>
,percentile,20
,n_estimators,500
,criterion,'gini'
,max_depth,40
,min_samples_split,2
,min_samples_leaf,2


In [12]:

# ----------------------
# Save everything
# ----------------------

# Save search best params separately for reference
joblib.dump(best_params, hyperparams_path.replace('.pkl', '_search.pkl'))

# Save final hyperparameters — store rf__ prefixed params plus fs percentile
# so the format matches what was saved from the search step
final_params = {f'rf__{k}': v for k, v in rf_params.items()}
final_params['feature_selection__percentile'] = fs_percentile
joblib.dump(final_params, hyperparams_path)

# Save selected features
selected_mask = final_pipeline.named_steps['feature_selection'].get_support()
selected_features = X_train.columns[selected_mask].tolist()
with open(selected_features_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")

# Save the whole pipeline
joblib.dump(final_pipeline, pipeline_path)

# Save model separately as well for easy loading without the feature selection step if needed
model_path = os.path.join(output_dir, "rf_model.pkl")
joblib.dump(final_pipeline.named_steps['rf'], model_path)

print(f"Saved hyperparameters to {hyperparams_path}")
print(f"Saved {len(selected_features)} selected features to {selected_features_path}")
print(f"Saved final pipeline to {pipeline_path}")
print(f"Saved final RF model to {model_path}")

Saved hyperparameters to /home/mariya/Data/pallas/models/gex/hyperparams.pkl
Saved 3923 selected features to /home/mariya/Data/pallas/models/gex/selected_features.txt
Saved final pipeline to /home/mariya/Data/pallas/models/gex/pipeline.pkl
Saved final RF model to /home/mariya/Data/pallas/models/gex/rf_model.pkl
